# Extract NASADEM, HydroSHEDS Basins & ERA5-Land Precipitation for a Land Cover Raster Footprint

This notebook uses **geemap** and the **Google Earth Engine (GEE)** Python API to:

1. Read an input land cover raster (e.g. a MAES / CORINE-style classification) and determine its spatial footprint.
2. Build an Earth Engine geometry from that footprint (exact valid-data footprint, or a simple bounding box).
3. Query and clip three datasets to that footprint:
   - **`NASA/NASADEM_HGT/001`** — elevation (DEM)
   - **`WWF/HydroSHEDS/v1/Basins/hybas_12`** — level-12 hydrological sub-basins (vector)
   - **`ECMWF/ERA5_LAND/MONTHLY_AGGR`** — annual total precipitation, summed from monthly totals for a chosen year
4. Export each layer, with the two raster layers (DEM, precipitation) optionally resampled to match the input raster's grid (CRS, resolution, transform) exactly — useful for stacking as covariates in downstream modeling (e.g. InVEST, terrain- or climate-corrected vegetation indices). The basins are exported as a vector file (GeoPackage).
5. Do a quick interactive visual QA with geemap.

**Requirements:** `geemap`, `earthengine-api`, `rasterio`, `geopandas`, `shapely`, `numpy`. An authenticated Earth Engine account/project is required.

## 1. Setup

In [ ]:
# Uncomment to install (first run only)
# %pip install -q geemap earthengine-api rasterio geopandas shapely numpy

In [21]:
import os

# ------------------------------------------------------------------
# Avoid PROJ "DATABASE.LAYOUT.VERSION" mismatches
# ------------------------------------------------------------------
# Root cause: modern pyproj / rasterio wheels each ship their OWN
# self-contained PROJ (a compiled libproj matched to its own proj.db
# schema version). The crash happened because something in this
# environment (the JupyterLab Desktop app bundles its own, much older
# PROJ under .../jlab_server/share/proj) had already set PROJ_LIB /
# PROJ_DATA / GDAL_DATA to point at a foreign proj.db. Manually
# repointing them at pyproj's data dir (the previous fix attempt)
# didn't help either, since rasterio's compiled libproj is newer than
# the proj.db that pyproj ships - still a mismatch, just a different one.
#
# Fix: strip any inherited PROJ_LIB / PROJ_DATA / GDAL_DATA *before*
# importing rasterio, then import rasterio first. Rasterio wheels
# auto-configure GDAL_DATA/PROJ_LIB to their own bundled, version-
# matched PROJ data on import - but only if nothing has already set
# those variables. pyproj is left alone and will independently fall
# back to its own bundled data too (both are self-contained on PyPI
# wheels, so neither needs to borrow the other's proj.db).
for _var in ("PROJ_LIB", "PROJ_DATA", "GDAL_DATA"):
    os.environ.pop(_var, None)

import rasterio  # import BEFORE pyproj/ee/geemap so its self-contained
                  # GDAL/PROJ configuration wins and nothing else can
                  # inject a conflicting proj.db first
from rasterio.crs import CRS
from rasterio.warp import transform_bounds
from rasterio.features import shapes

import pyproj
import ee
import geemap
import geopandas as gpd
import numpy as np
from shapely.geometry import box, shape, mapping

print("Rasterio:", rasterio.__version__)
print("Rasterio's PROJ_LIB (auto-set):", os.environ.get("PROJ_LIB"))
print("PyProj:", pyproj.__version__)
print("PyProj data dir:", pyproj.datadir.get_data_dir())

# Test that EPSG lookup works
print(CRS.from_epsg(4326))
print(CRS.from_epsg(6383))

Rasterio: 1.5.0
Rasterio's PROJ_LIB (auto-set): None
PyProj: 3.7.2
PyProj data dir: /Users/gregorygiuliani/Library/jupyterlab-desktop/jlab_server/envs/gee/lib/python3.14/site-packages/pyproj/proj_dir/share/proj
EPSG:4326
EPSG:6383


### Earth Engine authentication

Replace `EE_PROJECT` with your Google Cloud project registered for Earth Engine access. If you have already run `ee.Authenticate()` before on this machine, the cached credentials will be reused.

In [22]:
EE_PROJECT = "geemap-496017"  # <-- set your GEE-enabled Cloud project

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialized.")

Earth Engine initialized.


## 2. Configuration

In [ ]:
# --- Input / output paths -----------------------------------------------
INPUT_RASTER   = "/Users/gregorygiuliani/Desktop/input/LE_D26_kyiv_2017_maesL2.tif"            # path to your land cover raster
OUTPUT_DEM     = "/Users/gregorygiuliani/Desktop/outputs/nasadem_clip.tif"       # elevation output
OUTPUT_PRECIP  = "/Users/gregorygiuliani/Desktop/outputs/era5land_precip_clip.tif"  # annual precipitation output
OUTPUT_BASINS  = "/Users/gregorygiuliani/Desktop/outputs/hybas12_clip.gpkg"      # HydroSHEDS basins output (vector)

# If INPUT_RASTER has no CRS embedded in its metadata (src.crs is None),
# this value is used instead. Leave as None to rely entirely on the
# raster's own metadata; set e.g. "EPSG:6875" if you know the correct CRS
# but it isn't being picked up automatically.
INPUT_CRS_OVERRIDE = 'EPSG:6383'

# --- Footprint extraction strategy ---------------------------------------
# "bbox"  -> use the rectangular bounding box of the raster (fast, simple)
# "exact" -> dissolve the footprint of all valid (non-nodata) pixels into a
#            polygon (slower, but avoids fetching data for areas outside the
#            actual data extent, e.g. irregular / rotated / clipped rasters)
FOOTPRINT_MODE = "bbox"

# Optional buffer (in the input raster's CRS units) applied to the footprint
# before querying the datasets. Set to 0 to disable. A small buffer is
# recommended for the basins layer so that sub-basins straddling the edge
# of the land cover extent aren't clipped away.
BUFFER_DISTANCE = 0

# --- Export options --------------------------------------------------------
# If True, exported raster layers (DEM, precipitation) are resampled/
# reprojected to exactly match the input raster's CRS, resolution, and pixel
# grid - convenient for direct stacking as covariate bands. Does not apply
# to the HydroSHEDS basins, which are always exported as vector features.
MATCH_INPUT_GRID = True

# Native resolutions in metres, used when MATCH_INPUT_GRID is False.
NATIVE_DEM_SCALE     = 30     # NASADEM, ~1 arc-second
NATIVE_PRECIP_SCALE  = 11132  # ERA5-Land, ~0.1 degree

# --- ERA5-Land precipitation options ---------------------------------------
# Calendar year over which the monthly total_precipitation_sum band is
# summed to obtain annual precipitation. ERA5-Land Monthly Aggregated
# ("ECMWF/ERA5_LAND/MONTHLY_AGGR") is available from Feb 1950 to ~3 months
# before present.
PRECIP_YEAR = 2020

os.makedirs(os.path.dirname(OUTPUT_DEM) or ".", exist_ok=True)

# --- Google Drive export options -------------------------------------------
# geemap.ee_export_image_to_drive() runs an Earth Engine *batch* export:
# it submits a server-side task that writes the result directly into your
# Google Drive, rather than streaming bytes back through the interactive
# getDownloadURL endpoint (which is what was hitting the ~48 MB /
# 10,000x10,000 px request limit before). Batch exports support much
# larger regions, but files land in Drive, not on local disk - download
# them (or sync the folder) before running the alignment check in
# section 9 below.
DRIVE_FOLDER = "gee_exports_kyiv"   # created automatically if it doesn't exist

# If True, block and poll Earth Engine every WAIT_POLL_SECONDS until each
# export task finishes (or fails) before moving to the next one, printing
# progress along the way. If False, tasks are submitted and left running
# in the background - check progress at https://code.earthengine.google.com/tasks
WAIT_FOR_DRIVE_TASKS = True
WAIT_POLL_SECONDS = 20


## 3. Read the input raster footprint

In [24]:
with rasterio.open(INPUT_RASTER) as src:
    src_crs = src.crs
    src_transform = src.transform
    src_width, src_height = src.width, src.height
    src_bounds = src.bounds
    src_nodata = src.nodata
    src_res = src.res  # (x_res, y_res) in src_crs units

    if FOOTPRINT_MODE == "bbox":
        geom_native = box(*src_bounds)

    elif FOOTPRINT_MODE == "exact":
        # Build a mask of valid data and vectorize it into polygon(s)
        band1 = src.read(1)
        if src_nodata is not None:
            mask = band1 != src_nodata
        else:
            mask = np.ones(band1.shape, dtype=bool)

        geoms = [
            shape(geom)
            for geom, val in rasterio.features.shapes(
                mask.astype(np.uint8), mask=mask, transform=src_transform
            )
        ]
        if not geoms:
            raise ValueError("No valid-data pixels found to build an exact footprint.")
        geom_native = gpd.GeoSeries(geoms, crs=src_crs).unary_union

    else:
        raise ValueError("FOOTPRINT_MODE must be 'bbox' or 'exact'")

    if BUFFER_DISTANCE:
        geom_native = geom_native.buffer(BUFFER_DISTANCE)

print(f"CRS: {src_crs}")
print(f"Resolution: {src_res}")
print(f"Size: {src_width} x {src_height}")
print(f"Bounds ({src_crs}): {src_bounds}")

if src_crs is None:
    if INPUT_CRS_OVERRIDE is not None:
        src_crs = rasterio.crs.CRS.from_string(INPUT_CRS_OVERRIDE)
        print(f"INPUT_RASTER had no embedded CRS; using INPUT_CRS_OVERRIDE = {INPUT_CRS_OVERRIDE} instead.")
    else:
        raise ValueError(
            "INPUT_RASTER has no CRS defined (src.crs is None). "
            "This is the most common cause of \"Cannot transform naive "
            "geometries\" errors further down the notebook, since geopandas "
            "can't reproject a geometry it doesn't know the source CRS of.\n\n"
            "Fix options:\n"
            "  1. If you KNOW the correct CRS (e.g. from metadata/documentation), "
            "set INPUT_CRS_OVERRIDE above to it, e.g. INPUT_CRS_OVERRIDE = \"EPSG:6875\", "
            "and re-run from this cell.\n"
            "  2. Alternatively, permanently fix the file itself without altering "
            "pixel values:\n"
            "       import rasterio\n"
            "       with rasterio.open(INPUT_RASTER, 'r+') as f:\n"
            "           f.crs = 'EPSG:XXXX'  # replace with the correct EPSG code\n"
            "  3. If the raster is genuinely unprojected/unreferenced, georeference "
            "it first (e.g. in QGIS, or with rasterio/GDAL ground control points) "
            "before running this notebook."
        )

CRS: EPSG:6383
Resolution: (10.0, 10.0)
Size: 21130 x 26371
Bounds (EPSG:6383): BoundingBox(left=457800.0, bottom=5453430.0, right=669100.0, top=5717140.0)


In [25]:
# Reproject the footprint to WGS84 (EPSG:4326) for use with Earth Engine.
# Build the GeoDataFrame WITHOUT a crs kwarg, then set it explicitly via
# set_crs() - this avoids silently producing a naive (CRS-less) object if
# src_crs is ever an unexpected type (e.g. a string GDAL can parse but
# geopandas' constructor mishandles).
footprint_gdf = gpd.GeoDataFrame(geometry=[geom_native])
footprint_gdf = footprint_gdf.set_crs(src_crs, allow_override=True)

assert footprint_gdf.crs is not None, (
    "footprint_gdf still has no CRS after set_crs() - double-check that "
    "src_crs (printed in the previous cell) is a valid CRS."
)

footprint_gdf = footprint_gdf.to_crs("EPSG:4326")
geom_wgs84 = footprint_gdf.geometry.iloc[0]

footprint_ee = ee.Geometry(mapping(geom_wgs84))
print("Footprint reprojected to EPSG:4326.")
footprint_gdf

Footprint reprojected to EPSG:4326.


,geometry
0,"POLYGON ((32.05393 49.10223, 32.31105 51.46328..."


## 4. NASADEM — load and clip

In [26]:
nasadem = ee.Image("NASA/NASADEM_HGT/001").select("elevation")

dem_clipped = nasadem.clip(footprint_ee)

# Quick sanity check on the elevation range within the footprint
dem_stats = dem_clipped.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=footprint_ee,
    scale=NATIVE_DEM_SCALE,
    maxPixels=1e10,
    bestEffort=True,
).getInfo()
print("Elevation min/max (m):", dem_stats)

Elevation min/max (m): {'elevation_max': 339, 'elevation_min': 35}


## 5. HydroSHEDS level-12 basins — load and clip

`WWF/HydroSHEDS/v1/Basins/hybas_12` is a **vector** `FeatureCollection` of hydrological sub-basin polygons (Level 12, the finest HydroBASINS resolution), not an image, so it's handled differently from the two raster layers: we filter by spatial intersection with the footprint rather than clip a raster.

In [27]:
hybas12 = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_12")

# Keep only sub-basins that intersect the land cover footprint
basins_clipped = hybas12.filterBounds(footprint_ee)

n_basins = basins_clipped.size().getInfo()
print(f"HydroSHEDS level-12 sub-basins intersecting the footprint: {n_basins}")

HydroSHEDS level-12 sub-basins intersecting the footprint: 501


## 6. ERA5-Land annual precipitation — load, sum, and clip

`ECMWF/ERA5_LAND/MONTHLY_AGGR` provides monthly-aggregated bands; `total_precipitation_sum` is the monthly accumulated precipitation in **metres**. Annual total precipitation is obtained by summing the 12 monthly images for `PRECIP_YEAR`, then converting to millimetres.

In [28]:
era5_land_monthly = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

precip_year_ic = era5_land_monthly.filter(
    ee.Filter.calendarRange(PRECIP_YEAR, PRECIP_YEAR, "year")
).select("total_precipitation_sum")

n_months = precip_year_ic.size().getInfo()
if n_months == 0:
    raise ValueError(
        f"No ERA5-Land monthly images found for {PRECIP_YEAR}. "
        "Check that the year falls within the dataset's availability."
    )
elif n_months < 12:
    print(f"Warning: only {n_months}/12 months available for {PRECIP_YEAR} "
          "(likely the most recent, still-incomplete year).")

# Sum monthly totals (m) -> annual total, then convert to mm
precip_annual_m = precip_year_ic.sum()
precip_annual_mm = precip_annual_m.multiply(1000).rename("annual_precipitation_mm")

precip_clipped = precip_annual_mm.clip(footprint_ee)

precip_stats = precip_clipped.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=footprint_ee,
    scale=NATIVE_PRECIP_SCALE,
    maxPixels=1e10,
    bestEffort=True,
).getInfo()
print(f"Annual precipitation {PRECIP_YEAR} min/max (mm):", precip_stats)

Annual precipitation 2020 min/max (mm): {'annual_precipitation_mm_max': 667.95372394823, 'annual_precipitation_mm_min': 512.0170506646531}


## 7. Interactive visual QA

In [29]:
Map = geemap.Map()
Map.centerObject(footprint_ee, zoom=10)

elev_vis = {
    "min": float(dem_stats.get("elevation_min", 0)),
    "max": float(dem_stats.get("elevation_max", 2000)),
    "palette": [
        "0000ff", "00ffff", "00ff00", "ffff00", "ff8000", "ff0000", "ffffff",
    ],
}

precip_vis = {
    "min": float(precip_stats.get("annual_precipitation_mm_min", 0)),
    "max": float(precip_stats.get("annual_precipitation_mm_max", 2000)),
    "palette": [
        "ffffcc", "c7e9b4", "7fcdbb", "41b6c4", "1d91c0", "225ea8", "0c2c84",
    ],
}

Map.addLayer(dem_clipped, elev_vis, "NASADEM (clipped)")
Map.addLayer(precip_clipped, precip_vis, f"ERA5-Land annual precip {PRECIP_YEAR} (mm)", opacity=0.7)
Map.addLayer(
    basins_clipped.style(color="ff0000", fillColor="00000000", width=1.5),
    {},
    "HydroSHEDS hybas_12 (clipped)",
)
Map.addLayer(footprint_ee, {"color": "black"}, "Land cover footprint", opacity=0.3)
Map

Map(center=[50.33939643194369, 30.699418177786576], controls=(WidgetControl(options=['position', 'transparent_…

## 8. Export the layers

Rasters (DEM, precipitation): if `MATCH_INPUT_GRID = True`, each is downloaded aligned to the exact CRS, resolution, and pixel grid of `INPUT_RASTER`, so they can be stacked band-for-band with the land cover raster without any further resampling. If `False`, each is downloaded at its native scale, clipped to the footprint bounding box.

Basins: always exported as a vector file (GeoPackage) since `hybas_12` is a `FeatureCollection`, not an image.

In [30]:
import time


def _wait_for_task(task, label):
    """Poll an ee.batch.Task until it finishes, printing status changes."""
    if not WAIT_FOR_DRIVE_TASKS:
        print(f"  Submitted (not waiting): {label} -> task id {task.id}")
        return
    last_state = None
    while True:
        status = task.status()
        state = status.get("state")
        if state != last_state:
            print(f"  [{label}] {state}")
            last_state = state
        if state in ("COMPLETED", "FAILED", "CANCELLED"):
            if state == "FAILED":
                print(f"    error_message: {status.get('error_message')}")
            break
        time.sleep(WAIT_POLL_SECONDS)


def export_raster_to_drive(image, description, file_name_prefix, native_scale):
    """Submit a batch export of `image` to Google Drive, matching the input
    raster's grid exactly when MATCH_INPUT_GRID is True, otherwise at the
    dataset's native scale (clipped to the footprint bounding box)."""
    common_kwargs = dict(
        description=description,
        folder=DRIVE_FOLDER,
        fileNamePrefix=file_name_prefix,
        region=footprint_ee,
        maxPixels=1e13,
        fileFormat="GeoTIFF",
    )
    if MATCH_INPUT_GRID:
        common_kwargs.update(scale=src_res[0], crs=str(src_crs))
    else:
        common_kwargs.update(scale=native_scale)

    geemap.ee_export_image_to_drive(image, **common_kwargs)

    # geemap's helper starts the ee.batch task internally but doesn't hand
    # it back, so grab the most recently submitted task with this
    # description to monitor/report on it.
    matching = [
        t for t in ee.batch.Task.list()
        if t.config.get("description") == description
    ]
    task = matching[0] if matching else None
    print(f"Submitted Drive export: {description} "
          f"-> Drive:{DRIVE_FOLDER}/{file_name_prefix}.tif")
    if task is not None:
        _wait_for_task(task, description)
    else:
        print("  (could not locate task to monitor - check the Tasks tab)")


# --- DEM ---
export_raster_to_drive(dem_clipped, "NASADEM_export", "nasadem_clip", NATIVE_DEM_SCALE)

# --- Annual precipitation ---
export_raster_to_drive(precip_clipped, "ERA5Land_precip_export", "era5land_precip_clip", NATIVE_PRECIP_SCALE)

# --- HydroSHEDS basins (vector) ---
# EE table exports to Drive support CSV/GeoJSON/KML/KMZ/SHP/TFRecord, not
# GeoPackage directly. GeoJSON round-trips cleanly through geopandas, so
# export as GeoJSON and (optionally) convert to .gpkg locally after
# downloading - see the commented-out snippet below.
basins_description = "HydroSHEDS_hybas12_export"
basins_prefix = "hybas12_clip"
geemap.ee_export_vector_to_drive(
    basins_clipped,
    description=basins_description,
    folder=DRIVE_FOLDER,
    fileNamePrefix=basins_prefix,
    fileFormat="GeoJSON",
)
matching = [
    t for t in ee.batch.Task.list()
    if t.config.get("description") == basins_description
]
print(f"Submitted Drive export: {basins_description} "
      f"-> Drive:{DRIVE_FOLDER}/{basins_prefix}.geojson")
if matching:
    _wait_for_task(matching[0], basins_description)

# To convert the downloaded GeoJSON to GeoPackage locally once it's synced
# from Drive to OUTPUT_BASINS's folder:
#   import geopandas as gpd
#   gpd.read_file("<downloaded>.geojson").to_file(OUTPUT_BASINS, driver="GPKG")

Submitted Drive export: NASADEM_export -> Drive:gee_exports_mazovia/nasadem_clip.tif
  [NASADEM_export] READY
  [NASADEM_export] RUNNING
  [NASADEM_export] FAILED
    error_message: Google Drive folder not found
Submitted Drive export: ERA5Land_precip_export -> Drive:gee_exports_mazovia/era5land_precip_clip.tif
  [ERA5Land_precip_export] READY
  [ERA5Land_precip_export] RUNNING
  [ERA5Land_precip_export] COMPLETED
Exporting HydroSHEDS_hybas12_export... Please check the Task Manager from the JavaScript Code Editor.
Submitted Drive export: HydroSHEDS_hybas12_export -> Drive:gee_exports_mazovia/hybas12_clip.geojson
  [HydroSHEDS_hybas12_export] READY
  [HydroSHEDS_hybas12_export] COMPLETED


> **Drive exports:** these are asynchronous Earth Engine batch tasks, so they aren't subject to the ~48 MB / ~10,000x10,000 px `getDownloadURL` limit that direct `ee_export_image` downloads hit. Files are written to the `DRIVE_FOLDER` in your Google Drive rather than to local disk - sync or manually download `nasadem_clip.tif`, `era5land_precip_clip.tif`, and `hybas12_clip.geojson` into the `outputs/` folder (matching `OUTPUT_DEM`/`OUTPUT_PRECIP`/`OUTPUT_BASINS`) before running the alignment check in section 9. With `WAIT_FOR_DRIVE_TASKS = True` the cell above blocks until each task completes; with very large exports EE may also split output into multiple Drive files (e.g. `-0000000000-0000000000` tiles) if `fileDimensions` is exceeded.

## 9. (Optional) Verify raster alignment with the input raster

In [ ]:
if MATCH_INPUT_GRID:
    with rasterio.open(INPUT_RASTER) as lc_src:
        print("Land cover  -> CRS:", lc_src.crs, "| shape:", (lc_src.height, lc_src.width), "| transform:", lc_src.transform)

        for label, path in [("NASADEM", OUTPUT_DEM), ("ERA5-Land precip", OUTPUT_PRECIP)]:
            if os.path.exists(path):
                with rasterio.open(path) as r_src:
                    print(f"{label:<18}-> CRS:", r_src.crs, "| shape:", (r_src.height, r_src.width), "| transform:", r_src.transform)
                    same_crs = r_src.crs == lc_src.crs
                    same_shape_close = abs(r_src.height - lc_src.height) <= 1 and abs(r_src.width - lc_src.width) <= 1
                    print(f"    CRS match: {same_crs} | Shape approx. match: {same_shape_close}")